In [1]:
import torch
import math

In [2]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

In [3]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = torch.nn.Linear(d_model, d_ff)
        self.dropout = torch.nn.Dropout(dropout)
        self.linear2 = torch.nn.Linear(d_ff, d_model)
        self.relu = torch.nn.ReLU()

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [4]:
class TransformerEncoderBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _ = self.attention(query=x, key=x, value=x, attn_mask=mask)
        
        x = self.norm1(x + self.dropout(attn_out))
        
        ff_out = self.feed_forward(x)
        
        x = self.norm2(x + self.dropout(ff_out))
        return x

In [5]:
class CustomTransformerEncoder(torch.nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, num_layers=6, d_ff=2048, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        self.embedding_dropout = torch.nn.Dropout(dropout)
        
        self.layers = torch.nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])
        
        self.fc_out = torch.nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        x = self.pos_encoder(x)
        x = self.embedding_dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask)
            
        out = self.fc_out(x)
        return out

In [6]:
import time
import matplotlib.pyplot as plt

In [7]:
class NativeTransformerEncoder(torch.nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=4, d_ff=1024, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        self.embedding_dropout = torch.nn.Dropout(dropout)
        
        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model, num_heads, dim_feedforward=d_ff, dropout=dropout, batch_first=True
        )
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc_out = torch.nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.embedding_dropout(x)
        x = self.transformer(x, mask=mask)
        return self.fc_out(x)

In [8]:
vocab_size = 5000
d_model = 256
seq_len = 128
batch_size = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
custom_model = CustomTransformerEncoder(vocab_size, d_model=d_model).to(device)
native_model = NativeTransformerEncoder(vocab_size, d_model=d_model).to(device)

In [10]:
def benchmark_throughput(model, steps=50):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
    
    for _ in range(5):
        out = model(x)
        loss = criterion(out.view(-1, vocab_size), y.view(-1))
        loss.backward()
        
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start = time.time()
    
    for _ in range(steps):
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        
    if torch.cuda.is_available(): torch.cuda.synchronize()
    end = time.time()
    return (steps * batch_size) / (end - start)

In [11]:
print(f"Custom Throughput: {benchmark_throughput(custom_model):.2f} seq/s")
print(f"Native Throughput: {benchmark_throughput(native_model):.2f} seq/s")

Custom Throughput: 414.94 seq/s
Native Throughput: 873.03 seq/s
